<h1>Chapter 6 - Planning & Reflection</h1>
<i>More Autonomy for your `TinyAgent`</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 6 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 4`

At the beginning of every chapter, we start by choosing the LLM that we want to use. In this notebook, we will explore how to enable autonomous behavior for LLMs that have native tool calling and reasoning behavior. As such, the model that we will be using throughout this chapter is Gemma 4, a model with native tool calling and reasoning capabilities.

In [1]:
import os
from illustrated_agents.chapters.ch5_native import LLM

# Ollama through OpenAI API
llm = LLM(model="gemma4:e4b", backend="openai", api_base="http://localhost:11434/v1/", think=True)

# Llama.cpp server
# llm = LLM(model="openai/gemma-4-E4B-it-Q4_K_M", backend="litellm", api_base="http://localhost:8080", think=True)

# LM Studio
# llm = LLM(model="lm_studio/gemma-4-E4B-it", backend="litellm", api_base="http://localhost:1234/v1", think=True)

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash", backend="litellm", api_key=None)

## 2 - What does **`Autonomy`** look with Native Tool Calling and Reasoning?

In the previous chapter, we explored how to give your `TinyAgent` autonomy by explicit chains of `THOUGHT`, `OBSERVATION`, and `ACTION`. This worked quite well with a model that wasn't specifically trained for such autonomous behavior. We saw that this could be brittle needing to use regular expressions to extract those steps and have the underlying model execute actions. The parsing with JSON and XML, although a nice educational way of learning how it is done under the hood, is error-prone. 

Fortunately, we can use more recent models that were trained specifically for these tasks (reasoning and tool calling) as we did with native tool calling in Chapter 5 and native reasoning in Chapter 3. Let's explore how we can use newer models that have been trained to perform Reason and Act implicitly rather than explicitly as we did in `chapter06.ipynb`!

We previously implemented loops of:

* `THOUGHT` - A reasoning step about the current situation
* `ACTION` - An action to execute (e.g., a tool)
* `OBSERVATION` - A generated observation (typically the output of a tool)

However, with native tool calling there is no need for explicit `ACTION` and with native reasoning there is no need for explicit `THOUGHT`. We can replace what we already have for each of them with the following:

* `THOUGHT` -> Replace with `Response.reasoning` and `NativeReAct`
* `ACTION` -> Replace with `Response.tool_call` and `NativeTools`
* `OBSERVATION` -> Replace with adding the output of a tool to memory with `memory.add("tool", observation)`

Let's explore how we can do that starting with `NativeReAct`:




## 3 - "Building" `NativeReAct`

Since we can replace the idea of explicit `THOUGHT` / `ACTION` / `OBSERVATION` with native capabilities, the `ReAct` we used to have is not necessary anymore. Even moreso, there is only one functionality that needs to remain and that is the maximum number of steps each run can take. There is no need to instruct the model on `THOUGHT` / `ACTION` / `OBSERVATION` nor is parsing of them necessary. As such, we can create a `NativeReAct` class that simply does no processing to the `Response` nor has any `.prompt` that we will need to use:

In [2]:
from illustrated_agents.chapters.ch6 import ReAct

class NativeReAct(ReAct):
    """ReAct using native LLM reasoning instead of text-based parsing."""

    @property
    def prompt(self) -> str:
        return ""

    def parse(self, response):
        return response

And that's it! Really, there is nothing more that we need to add to use native reasoning and tool calling. The `NativeReAct` and `NativeTools` already handle all the "not-so-heavy" lifting.

## 4 - The Native `TinyAgent`

Usage of your newly fully native `TinyAgent` is the same as before but we use `NativeTools` instead of `Tools` and `NativeReAct` instead of `ReAct`. Let's start by initializing the `TinyAgent` first:

In [6]:
from illustrated_agents.toolbox import add, get_weather
from illustrated_agents.chapters.ch5_native import Memory, NativeTools
from illustrated_agents.chapters.ch6 import TinyAgent

# Native Tools
tools = NativeTools()
tools.add_tool("add", add)
tools.add_tool("get_weather", get_weather)

# Memory
memory = Memory()

# Native ReAct
react = NativeReAct(max_steps=10)

# Initialize Agent
agent = TinyAgent(
    llm=llm, 
    memory=memory, 
    tools=tools, 
    planner=react
)

Next up, we can run the same script we did as in `chapter06.ipynb`:

In [7]:
from rich import print as pprint

# Multi-step task with reasoning
output = agent.run("""
I'm planning a trip! Help me with these tasks:
1. What's the weather in New York City?
2. What's the weather in Los Angeles?
3. I saved $150.50 and my friend is giving me $75.25. How much do I have for the trip?

Based on the weather, which city would you recommend I visit?
""")
pprint(output)

Here is the information you requested for your trip planning!

**1. Weather in New York City:**
It is currently **Sunny and 72°F**.

**2. Weather in Los Angeles:**
It is currently **Sunny and 72°F**.

**3. Total Money for the Trip:**
You will have **$225.75** for your trip ($150.50 + $75.25).

**Weather Recommendation:**
Based on the weather, **both New York City and Los Angeles look equally wonderful!** Since both are reported to be 
sunny and 72°F, you can't go wrong.

*   **If you're looking for a mix of culture, history, and city buzz:** New York City is fantastic.
*   **If you're looking for beautiful beaches, a more relaxed, laid-back vibe, and sunshine:** Los Angeles might be
your pick!

Ultimately, since the weather is perfect in both, consider what *activities* you'd prefer to do! Have a wonderful 
trip!

What's interesting about this response is that it is much more involved which is not surprising considering Gemma 4 is significantly better than Gemma 3 and also trained to perform better agentic loops. That said, let's explore the differences in messages between a native tool calling / reasoning model and a non-native model:

In [8]:
from rich import print as pprint

pprint(agent.memory.messages)

[
    {'role': 'system', 'content': 'You are a helpful assistant.\n\n'},
    {
        'role': 'user',
        'content': "\nI'm planning a trip! Help me with these tasks:\n1. What's the weather in New York City?\n2. 
What's the weather in Los Angeles?\n3. I saved $150.50 and my friend is giving me $75.25. How much do I have for 
the trip?\n\nBased on the weather, which city would you recommend I visit?\n"
    },
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_pbdqg9ay',
                'function': {'arguments': '{"location":"New York City"}', 'name': 'get_weather'},
                'type': 'function',
                'index': 0
            }
        ]
    },
    {'role': 'tool', 'content': 'Weather in New York City: Sunny, 72°F'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_33d0hed5',
                'function': {'arguments': '{"location":"Los Angeles"}', 'name': 'get_weather'},
                'type': 'function',
                'index': 0
            }
        ]
    },
    {'role': 'tool', 'content': 'Weather in Los Angeles: Sunny, 72°F'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_cozuurde',
                'function': {'arguments': '{"a":"150.50","b":"75.25"}', 'name': 'add'},
                'type': 'function',
                'index': 0
            }
        ]
    },
    {'role': 'tool', 'content': '225.75'},
    {
        'role': 'assistant',
        'content': "Here is the information you requested for your trip planning!\n\n**1. Weather in New York 
City:**\nIt is currently **Sunny and 72°F**.\n\n**2. Weather in Los Angeles:**\nIt is currently **Sunny and 
72°F**.\n\n**3. Total Money for the Trip:**\nYou will have **$225.75** for your trip ($150.50 + 
$75.25).\n\n**Weather Recommendation:**\nBased on the weather, **both New York City and Los Angeles look equally 
wonderful!** Since both are reported to be sunny and 72°F, you can't go wrong.\n\n*   **If you're looking for a mix
of culture, history, and city buzz:** New York City is fantastic.\n*   **If you're looking for beautiful beaches, a
more relaxed, laid-back vibe, and sunshine:** Los Angeles might be your pick!\n\nUltimately, since the weather is 
perfect in both, consider what *activities* you'd prefer to do! Have a wonderful trip!"
    }
]

Note how the model uses assistant uses an additional `tool_calls` field to call a specific tool with a set of arguments. Moreover, we add the output of the tool to a specific `tool` role so that the model understand the effect of its tool call, which is essentially the same as the `OBSERVATION` step in explicit ReAct:


```json
[
    ...
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_msvwwane',
                'function': {'arguments': '{"location":"Los Angeles"}', 'name': 'get_weather'},
                'type': 'function',
                'index': 0
            }
        ]
    },
    {'role': 'tool', 'content': 'Weather in Los Angeles: Sunny, 72°F'},
    ...
]
```

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we went from an explicit `ReAct`-loop to an implicit one! Since we already had worked on native capabilities early on there were fortunately few changes that needed to be made:

In [1]:
from illustrated_agents.chapters.ch6_native import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py    ← Updated (Autonomy with native reasoning and tool calling.)                                    │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py ← New (Added the `NativeReAct` class)                                                           │
│ ├── toolbox.py                                                                                                  │
│ └── tools.py                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

There are many more things that we can still add to the `TinyAgent` and that were covered in this chapter. Up next is adding reflection as a technique for the `TinyAgent` to early on discover if its going in the right direction or not. This is a technique that is typically done explicitly and therefore fits nicely with the explicit `ReAct` and `Tools` forms of the `TinyAgent`. However, it can also be used with the native Agent you just built. 